# 🏗️ Notebook 1: Google Search — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/google-search
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A web-scale search engine: take a text query, return the most relevant pages in <200ms.

### Functional requirements
- Crawl the web, follow links, fetch pages.
- **Index** pages by the words they contain.
- Given a query, return the top-K most relevant documents.
- Rank by relevance + quality + personalization.
- Suggestions (typeahead) — see the typeahead lab.

### Non-functional
- **Huge scale**: 50 B+ pages indexed, trillions of postings.
- **Query volume**: ~8 B searches/day (~100 k QPS at peak). We derive this below rather
  than assert it — the peak is what sizes the serving fleet.
- **Low latency** for queries (<200 ms p99, end to end, including fan-out and merge).
- Reasonable **freshness**: news within minutes, long-tail within weeks. Freshness and
  corpus size pull against each other — see the recrawl budget in the capacity cell.

## Three pipelines

```
   ┌──────────┐   fetch  ┌──────────┐  parse  ┌──────────┐
   │ Crawler  │─────────▶│ Document │────────▶│ Indexer  │
   └──────────┘          │  Store   │         └──────────┘
        ▲                └──────────┘              │
        │ seed + discovered                        ▼
        │                                   ┌────────────┐
        │                                   │ Inverted   │
        │                                   │ Index      │
        │                                   └────────────┘
        │                                         ▲
        │                                         │ lookup
        │                                   ┌────────────┐
        └─────── Ranking/Quality signals ───│ Query Svc  │◀── user
                                            └────────────┘
```

Three *independent* pipelines:
1. **Crawling** — build a document corpus.
2. **Indexing** — preprocess docs into an inverted index.
3. **Serving** — answer queries against the index.


## Back-of-envelope — let the code do the math

Three numbers decide this architecture, and they point at three different systems:

1. **Corpus size** → how many index shards (a storage problem).
2. **Query peak** → how many replicas of each shard (a throughput problem).
3. **Recrawl rate** → how big the crawl fleet is (a bandwidth problem).

Change the inputs and watch which one moves.

One subtlety worth naming before you read the output: with **document sharding**, every
query touches *every* shard. So shard count is set by corpus size, but the number of
machines is `shards × replicas`, and replicas are set by QPS. Those two multiply.

In [ ]:
# Capacity estimation for a web-scale search engine.
# All numbers are rough - the point is to practice thinking in orders of magnitude.
import math

PAGES            = 50_000_000_000    # 50B pages indexed
AVG_PAGE_KB      = 100               # average raw HTML size
INDEX_RATIO      = 0.15              # compressed inverted index ~ 15% of raw
SEARCHES_PER_DAY = 8_000_000_000     # ~8B searches/day
PEAK_RATIO       = 1.1               # peak QPS / average QPS (search is a 24h global load)
AVG_DOC_BYTES_RETURNED = 500         # snippet + url + title in a result
RESULTS_PER_QUERY = 10
SHARD_GB         = 500               # index bytes one serving box holds in RAM+SSD
SHARD_QPS        = 500               # queries/sec one replica of one shard can absorb
RECRAWL_DAYS     = 14                # average time to revisit a page

# ---- corpus + index ----
raw_pb   = PAGES * AVG_PAGE_KB * 1024 / (1024**5)
index_pb = raw_pb * INDEX_RATIO
shards   = math.ceil(index_pb * 1024**5 / (SHARD_GB * 1024**3))

# ---- query load ----
avg_qps  = SEARCHES_PER_DAY / 86_400
peak_qps = avg_qps * PEAK_RATIO
replicas = math.ceil(peak_qps / SHARD_QPS)     # every query hits every shard
egress_gbps = peak_qps * RESULTS_PER_QUERY * AVG_DOC_BYTES_RETURNED * 8 / 1e9

# ---- crawl load ----
fetches_s   = PAGES / (RECRAWL_DAYS * 86_400)
crawl_gbps  = fetches_s * AVG_PAGE_KB * 1024 * 8 / 1e9

print(f'Raw HTML corpus  : {raw_pb:>10,.1f} PB')
print(f'Inverted index   : {index_pb*1024:>10,.0f} TB  (compressed, {INDEX_RATIO:.0%} of raw)')
print(f'Index shards     : {shards:>10,}     (at {SHARD_GB} GB/shard)')
print()
print(f'Average QPS      : {avg_qps:>10,.0f}')
print(f'Peak QPS         : {peak_qps:>10,.0f}')
print(f'Replicas / shard : {replicas:>10,}     (every query fans out to every shard)')
print(f'Serving machines : {shards*replicas:>10,}     = shards x replicas')
print(f'Query egress     : {egress_gbps:>10,.1f} Gbps at peak')
print()
print(f'Recrawl rate     : {fetches_s:>10,.0f} pages/s  (whole corpus every {RECRAWL_DAYS} days)')
print(f'Crawl ingress    : {crawl_gbps:>10,.1f} Gbps sustained')

### Reading the output

- **~700 TB of index → ~1,400 shards.** Corpus size alone fixes the fan-out width, and
  fan-out width is what makes p99 latency hard: a query is only as fast as its slowest
  of 1,400 shards. This is the tail-at-scale problem, and it is why real systems send
  duplicate ("hedged") requests to a second replica after a few milliseconds.
- **~100 k peak QPS is the easy number.** It needs a couple hundred replicas per shard,
  which is a lot of machines but no cleverness.
- **The two multiply.** `shards × replicas` is the fleet. Halving the index size halves
  the fleet just as effectively as halving the QPS — which is why index compression is
  a first-class ranking-team concern, not a storage footnote.
- **Crawling is a bandwidth problem, serving is a latency problem.** A 14-day recrawl of
  50 B pages is ~41 k fetches/second and ~34 Gbps *sustained, forever*. Halving
  `RECRAWL_DAYS` to get fresher results doubles that bill. Freshness is not free; that
  is why real crawlers recrawl by *predicted change rate*, not uniformly.

## Tiny end-to-end search - the whole system in 20 lines

Before zooming into each pipeline, here is the **entire** crawl -> index -> query flow on 4 pages.
Every later notebook replaces one of these steps with a **better** version.


In [ ]:
# Minimum-viable search: crawl (mock), index, query.
from collections import defaultdict
import re

# 1) 'Crawl' - normally HTTP fetches; here we pretend.
CORPUS = {
    'https://a.com/py':    'Python is a popular programming language',
    'https://a.com/flask': 'Flask is a lightweight Python web framework',
    'https://b.com/django':'Django is a batteries-included Python web framework',
    'https://b.com/rust':  'Rust is a fast systems programming language',
}

# 2) Index - tokenize + build term -> set(doc_id)
def tok(t): return re.findall(r'[a-z]+', t.lower())
index = defaultdict(set)
docs = list(CORPUS.items())                   # doc_id = position in list
for doc_id, (_, text) in enumerate(docs):
    for w in tok(text):
        index[w].add(doc_id)

# 3) Query - intersect postings, return URLs
def search(q):
    terms = tok(q)
    if not terms: return []
    ids = set.intersection(*(index.get(t, set()) for t in terms))
    return [docs[i][0] for i in ids]

print('python web framework ->', search('python web framework'))
print('programming language ->', search('programming language'))
print('javascript           ->', search('javascript'))  # empty result


Everything from here is about making **each of those three lines of code** survive the
real internet: billions of pages, noisy text, hostile servers, and sub-second queries.
